# RDF-to-LPG Converter — Fuseki TBox → FalkorDB property graph

Implements the [RDF-to-LPG Converter specification](../../../docs/LPG%20Ontology%20Coverter/rdf-to-lpg-converter-specification.md):
replicate the TBox (classes, taxonomy, object properties) from **Fuseki** into a **FalkorDB**
labeled property graph, with concept embeddings for GraphRAG vector search alongside
sub-millisecond traversal.

This is the `ontology_modeler.lpg` subpackage, following the **Strategy Pattern**:

| Stage | Class | Does |
|-------|-------|------|
| **Extraction** | `TboxExtractor` | SPARQL queries A/B/C over the shared `FusekiClient` |
| **Transformation** | `MetaGraphBuilder` | builds a `networkx.MultiDiGraph` IR + concept profiles |
| **Embedding** | `get_embedder` | hashing (default, no deps) or sentence-transformers |
| **Loading** | `FalkorDBExporter` | parameterised Cypher `MERGE`/`UNWIND` into FalkorDB |

`OntologyModeler().lpg_converter()` (an `LpgConverter`) wires them together, sharing the
component's Fuseki and FalkorDB connections.

> The spec's Section 6 Redis schema-pruning cache is intentionally **not** built.

**Prerequisites:** Fuseki (with FIBO) and FalkorDB both up; run from
`src/Ontology Modeler/notebooks/`. This **clears and rebuilds** the FalkorDB `ontology`
graph (a derived replica); Fuseki is only read.

In [1]:
import sys, logging, time
from pathlib import Path
MODULE_DIR = Path.cwd().parent
if str(MODULE_DIR) not in sys.path:
    sys.path.insert(0, str(MODULE_DIR))

from ontology_modeler import (OntologyModeler, TboxExtractor, MetaGraphBuilder,
                              concept_profile, get_embedder, FalkorDBExporter)

logging.basicConfig(level=logging.INFO, format="%(message)s")
logging.getLogger("ontology_modeler.lpg").setLevel(logging.INFO)

om = OntologyModeler()
om.fuseki.ping()
FalkorDBExporter(om.falkor).ping()
print("Fuseki and FalkorDB both reachable")

Fuseki and FalkorDB both reachable


## 1. Structural translation model

| W3C Semantic Web (RDF) | Labeled Property Graph (LPG) |
|------------------------|------------------------------|
| `owl:Class` | Node `:Class` |
| `rdfs:subClassOf` | Edge `-[:SUBCLASS_OF]->` |
| `owl:ObjectProperty` (domain→range) | Edge `-[:{PROPERTY_NAME}]->` |
| `rdfs:label` / `skos:prefLabel` | `name` |
| `skos:definition` / `rdfs:comment` | `definition` |
| `skos:altLabel` | `alt_labels` (string[]) |
| concept profile → vector | `embedding` (float[]) |

Direction: `(subclass)-[:SUBCLASS_OF]->(superclass)`, so traversing *out* walks up.

## 2. Extraction — three SPARQL queries

`TboxExtractor` runs queries A (classes+labels), B (subclass taxonomy, minus `owl:Thing`),
C (object properties with named domain/range), over the shared client.

In [2]:
ex = TboxExtractor(om.fuseki)

t = time.monotonic(); classes = ex.classes()
print(f"Query A — classes:              {len(classes):5d}   ({time.monotonic()-t:.1f}s)")
taxonomy = ex.taxonomy();  print(f"Query B — subclass edges:       {len(taxonomy):5d}")
props = ex.object_properties(); print(f"Query C — object-property edges:{len(props):5d}")

c = next(c for c in classes if (c.name or "") == "deposit account")
print("\nsample ClassRecord:")
print("  short_name:", c.short_name)
print("  name      :", c.name)
print("  definition:", (c.definition or "")[:70], "...")

Query A — classes:               3017   (0.1s)


Query B — subclass edges:        4212
Query C — object-property edges:  560

sample ClassRecord:
  short_name: ClientsAndAccounts:DepositAccount
  name      : deposit account
  definition: account that provides a record of money placed with a depository insti ...


In [3]:
p = next(p for p in props if (p.name or "") == "has contract party")
print("sample PropertyEdge:")
print("  rel_type :", p.rel_type, " (becomes the Cypher edge type)")
print("  domain   :", p.domain_iri.split('/')[-1], "-> range:", p.range_iri.split('/')[-1])

sample PropertyEdge:
  rel_type : HASCONTRACTPARTY  (becomes the Cypher edge type)
  domain   : Contract -> range: ContractParty


## 3. Transformation — the NetworkX IR

`MetaGraphBuilder` turns the extract into a `networkx.MultiDiGraph`. Two honest reductions,
both matching the loader's Cypher semantics: **node dedup** (Query A's `GROUP BY` yields
multiple rows for a class with several labels/definitions; they collapse to one node keyed by
IRI) and **endpoint filtering** (an edge whose other end wasn't captured is dropped, as the
Cypher `MATCH`es both ends).

In [4]:
builder = MetaGraphBuilder()
rows_in = builder.add_classes(classes)            # returns rows processed
nt = builder.add_taxonomy(taxonomy)
npr = builder.add_object_properties(props)
unique_nodes = builder.stats()["nodes"]           # distinct IRIs after dedup

print(f"class records in : {rows_in}")
print(f"unique nodes     : {unique_nodes}   (collapsed {rows_in-unique_nodes} multi-label/def duplicates)")
print(f"taxonomy edges   : {nt}   (dropped {len(taxonomy)-nt} with an uncaptured endpoint)")
print(f"property edges   : {npr}   (dropped {len(props)-npr} with an uncaptured endpoint)")
print("\nIR stats:", builder.stats())

class records in : 3017


unique nodes     : 3005   (collapsed 12 multi-label/def duplicates)
taxonomy edges   : 3843   (dropped 369 with an uncaptured endpoint)
property edges   : 468   (dropped 92 with an uncaptured endpoint)

IR stats: {'nodes': 3005, 'edges': 4311, 'subclass_edges': 3843, 'object_property_edges': 468}


In [5]:
dep_iri = "https://spec.edmcouncil.org/fibo/ontology/FBC/ProductsAndServices/ClientsAndAccounts/DepositAccount"
print(concept_profile(builder, dep_iri))

Class: deposit account. Definition: account that provides a record of money placed with a depository institution for safekeeping and management. Superclasses: banking product, investment or deposit account.


## 4. Embedding — pluggable, deps-optional

Each concept profile becomes a vector. **`HashingEmbedder`** (default) is deterministic
character-n-gram hashing, 384-d, no model. **`SentenceTransformerEmbedder`** (optional) is
real `all-MiniLM-L6-v2`, import-guarded so torch stays opt-in. Swap:
`get_embedder("sentence-transformers")`.

In [6]:
emb = get_embedder("hash")
vec = emb.encode(concept_profile(builder, dep_iri))
print(f"embedder: {type(emb).__name__}, dim={emb.dim}")
print(f"L2 norm: {sum(x*x for x in vec)**0.5:.3f} (normalised); deterministic: {emb.encode('x')==emb.encode('x')}")

embedder: HashingEmbedder, dim=384
L2 norm: 1.000 (normalised); deterministic: True


## 5. Loading — Cypher `MERGE` into FalkorDB

`FalkorDBExporter` ingests with `UNWIND`-batched, parameterised `MERGE` (idempotent).
Object-property edges need a dynamic relationship type: the token is sanitised at extraction
and re-validated (`^[A-Z][A-Z0-9_]*$`) before injection; every value stays a bound parameter.

`LpgConverter.run` does the whole pipeline; `clear=True` wipes the derived replica first.

In [7]:
conv = om.lpg_converter()
stats = conv.run(clear=True, embed=True, embedder="hash")
print("\nIR    :", stats["ir"])
print("Loaded:", stats["loaded"])
assert stats["loaded"]["nodes"] == stats["ir"]["nodes"]
assert stats["loaded"]["edges"] == stats["ir"]["edges"]
print("\n✓ FalkorDB graph is structurally identical to the IR")

extracted 3017 classes, 4212 subclass, 560 object-property rows


IR: 3005 nodes, 3843 subclass + 468 property edges


embedding 3005 concept profiles with HashingEmbedder (dim=384)


clearing FalkorDB graph 'ontology'


set 3005 embeddings; vector index created



IR    : {'nodes': 3005, 'edges': 4311, 'subclass_edges': 3843, 'object_property_edges': 468}
Loaded: {'nodes': 3005, 'edges': 4311, 'classes_merged': 3005, 'taxonomy_merged': 3843, 'properties_merged': 468, 'embeddings_set': 3005}

✓ FalkorDB graph is structurally identical to the IR


### Verifying edge counts

FalkorDB **under-counts** `count(*)` on an all-anonymous `MATCH ()-[r]->()` pattern;
`count(r)` is authoritative. `FalkorDBExporter.count_edges` uses `count(r)`.

In [8]:
exp = FalkorDBExporter(om.falkor)
qs = lambda c: exp.graph.query(c).result_set[0][0]
print("count(*)  (under-counts):", qs("MATCH ()-[r]->() RETURN count(*)"))
print("count(r)  (correct)     :", qs("MATCH ()-[r]->() RETURN count(r)"))
print("nodes                   :", exp.count_nodes())
print("SUBCLASS_OF edges       :", exp.count_edges("SUBCLASS_OF"))

count(*)  (under-counts): 4279
count(r)  (correct)     : 4311
nodes                   : 3005
SUBCLASS_OF edges       : 3843


## 6. GraphRAG — traversal + vector search

**(a) Topological traversal** — full ancestry via a variable-length `SUBCLASS_OF` path.

In [9]:
res = exp.graph.query(
    "MATCH (c:Class {iri:$iri})-[:SUBCLASS_OF*1..8]->(a:Class) RETURN DISTINCT a.name AS name",
    {"iri": dep_iri})
print("Deposit Account ancestry (LPG traversal):")
for r in res.result_set:
    print("   →", r[0])

Deposit Account ancestry (LPG traversal):
   → investment or deposit account
   → account
   → banking product
   → financial product
   → product


**(b) Vector search** — the pipeline created a vector index on `:Class(embedding)`. Embed a
phrase with the *same* embedder and ask for nearest concepts. (Hashing is lexical; with
sentence-transformers these become true semantic matches.)

In [10]:
for query in ["a bank account for holding deposited money",
              "a company that issues securities",
              "interest rate benchmark"]:
    q = get_embedder("hash").encode(query)
    print(f"\nKNN  “{query}”:")
    for name, score in exp.knn(q, k=3):
        print(f"    {score:.3f}  {name}")


KNN  “a bank account for holding deposited money”:
    0.485  deposit account
    0.584  Hong Kong depositary receipt
    0.589  investment or deposit account

KNN  “a company that issues securities”:
    0.493  depositary receipt
    0.506  announce securities issue
    0.519  securities underwriting issuance

KNN  “interest rate benchmark”:
    0.286  specific-provider interest rate benchmark
    0.385  interest rate reset
    0.417  floating interest rate


## 7. Idempotency

In [11]:
c2 = om.lpg_converter(); c2.extract_transform()
before = (exp.count_nodes(), exp.count_edges())
c2.load(clear=False, embeddings=None)
after = (exp.count_nodes(), exp.count_edges())
print("before re-load:", before, "| after:", after, "| idempotent:", before == after)

extracted 3017 classes, 4212 subclass, 560 object-property rows


IR: 3005 nodes, 3843 subclass + 468 property edges


before re-load: (3005, 4311) | after: (3005, 4311) | idempotent: True


## Summary — and the agent tool

**Runs live:** extraction (SPARQL A/B/C) → NetworkX IR → concept profiles → embeddings →
Cypher `MERGE` into FalkorDB → vector index → KNN + traversal. FIBO gives **3,005 class
nodes, 3,843 `SUBCLASS_OF`, 468 object-property edges**.

```python
from ontology_modeler import OntologyModeler, FalkorDBExporter, get_embedder
om = OntologyModeler()
om.lpg_converter().run(clear=True, embed=True)     # rebuild the LPG

def tool_find_concepts(text: str, k: int = 5) -> list[dict]:
    "Find ontology classes semantically nearest to a phrase (Entity Linking)."
    exp = FalkorDBExporter(om.falkor)
    return [{"name": n, "score": s} for n, s in exp.knn(get_embedder("hash").encode(text), k)]
```
CLI: `python -m ontology_modeler lpg --clear --embed` (add `--embedder sentence-transformers`
for real semantics). **Omitted:** the Redis schema-pruning cache (spec Section 6).